# Updated structural GoG embedding — Colab
Use a fresh GPU runtime and run the cells in order. This notebook reads the existing GoG JSON; it does not rebuild graphs or change splits.

Download the accompanying **gog_embedding_updated.zip** and place it, plus **gog_model_subset_coarse.json**, at the top level of My Drive. The code archive contains the local model updates, which are not yet committed/pushed to GitHub. A plain GitHub clone can therefore load older code.

The real run compares baseline continuation, R-GCN, legacy GoG and updated structural GoG with identical ROOT targets/candidates. Defaults are one seed (17), 80 shared baseline warm-up updates, 80 continuation/adaptation updates, and 160 updates for the independently trained R-GCN/legacy GoG. Validation only: no test metrics.

Source references: [Colab file/Drive I/O](https://colab.research.google.com/notebooks/io.ipynb), [PyG minimal installation](https://pytorch-geometric.readthedocs.io/en/latest/notes/installation.html).

## Cell 1 — GPU and dependencies

In [ ]:
import sys, subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then rerun."
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

# Preserve Colab's installed PyTorch/CUDA build.
constraints = Path("/content/gog_torch_constraints.txt")
constraints.write_text("torch==" + torch.__version__ + "\n")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch-geometric>=2.5", "pandas>=2.0,<3",
    "-c", str(constraints),
])

## Cell 2 — Drive and run configuration

In [ ]:
from google.colab import drive
from datetime import datetime, timezone

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
CODE_ZIP = DRIVE / "gog_embedding_updated.zip"
DATASET_NAME = "gog_model_subset_coarse.json"
DATA_SOURCE = DRIVE / DATASET_NAME

SEEDS = [17]                   # Later: [17, 23, 42].
WARMUP_UPDATES = 80
CONTINUATION_UPDATES = 80
HIDDEN_DIM = 128
NEGATIVES = 50
BATCH_SIZE = 4096              # If CUDA runs out of memory, use 1024.
ABLATIONS = []                 # Optional: ["identity_off", "chemical_off"].

REPO = Path("/content/updated_gog")
LOCAL_DATA = Path("/content/gog_inputs") / DATASET_NAME
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
OUTPUT_DIR = DRIVE / "gog_embedding_runs" / (
    "structural_seeds-" + "_".join(map(str, SEEDS)) + "_" + stamp
)
print("Code archive:", CODE_ZIP)
print("GoG input:", DATA_SOURCE)
print("Output:", OUTPUT_DIR)

## Cell 3 — Load and verify the updated code; copy data locally

In [ ]:
import hashlib, json, shutil, zipfile, os

for path in (CODE_ZIP, DATA_SOURCE):
    if not path.is_file():
        raise FileNotFoundError(
            f"Upload {path.name} to the top level of My Drive, or edit its path in Cell 2."
        )

REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as archive:
    if "bundle_manifest.json" not in archive.namelist():
        raise ValueError("Use the supplied updated-code archive.")
    for member in archive.infolist():
        destination = (REPO / member.filename).resolve()
        if not destination.is_relative_to(REPO.resolve()):
            raise ValueError("Invalid archive path")
    archive.extractall(REPO)

bundle = json.loads((REPO / "bundle_manifest.json").read_text())
for name, expected_sha in bundle["files"].items():
    actual = hashlib.sha256((REPO / name).read_bytes()).hexdigest()
    if actual != expected_sha:
        raise ValueError(f"Code checksum mismatch: {name}")

LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(DATA_SOURCE, LOCAL_DATA)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print(f"Verified {len(bundle['files'])} source files.")
print("Using local data copy:", LOCAL_DATA)

## Cell 4 — Validate the frozen dataset and trainer

In [ ]:
from data_pipeline.preprocessing import load_gog
from training.link_prediction import selected_facts, TrainingConfig
import training.experiments as experiments

assert Path(experiments.__file__).resolve().is_relative_to(REPO), (
    "A different code copy is already imported. Restart the runtime and run the cells in order."
)
assert "encoder_profile" in TrainingConfig.__dataclass_fields__

data = load_gog(LOCAL_DATA)
assert data.report["validation"] == "passed"
assert data.report["splits_generated"] is False, "Expected the supplied frozen splits."
assert any(g["kind"] == "internal" for g in data.graphs), (
    "Use the full structural file, not the ROOT-only ablation."
)

print("Entities:", len(data.entities))
print("Internal graphs:", sum(g["kind"] == "internal" for g in data.graphs))
print("Training message-passing edges:", sum(map(len, data.edge_lists.values())))
for split in ("train", "validation", "test"):
    print(f"ROOT {split} facts:", len(selected_facts(data, split, "root")))
print("Test counts are bookkeeping only; training below does not evaluate test metrics.")
del data

## Cell 5 — Bounded CPU smoke check

In [ ]:
# Quick implementation check on a tiny synthetic fixture; this is not a benchmark.
SMOKE_DIR = Path("/content/gog_smoke") / stamp
subprocess.run([
    sys.executable, "-u", "-m", "training.experiments",
    str(REPO / "docs/synthetic_gog_fixture.json"),
    "--device", "cpu", "--threads", "2",
    "--hidden-dim", "16", "--num-bases", "2",
    "--seeds", "17", "--warmup-updates", "2", "--continuation-updates", "2",
    "--negatives", "2", "--batch-size", "4",
    "--output-dir", str(SMOKE_DIR),
], cwd=REPO, check=True)
print("Smoke check passed.")

## Cell 6 — Train the real comparison on GPU

In [ ]:
# Full, single-seed validation comparison. No --evaluate-test flag.
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
shutil.copy2(REPO / "bundle_manifest.json", OUTPUT_DIR / "code_bundle_manifest.json")
command = [
    sys.executable, "-u", "-m", "training.experiments", str(LOCAL_DATA),
    "--encoder-profile", "structural", "--task-scope", "root",
    "--feature-mode", "categorical", "--rounds", "1",
    "--device", "cuda", "--threads", "2",
    "--seeds", *map(str, SEEDS),
    "--warmup-updates", str(WARMUP_UPDATES),
    "--continuation-updates", str(CONTINUATION_UPDATES),
    "--hidden-dim", str(HIDDEN_DIM),
    "--num-bases", "8", "--num-heads", "4", "--dropout", "0.2",
    "--negatives", str(NEGATIVES), "--batch-size", str(BATCH_SIZE),
    "--learning-rate", "0.001", "--weight-decay", "0.0001", "--gradient-clip", "1",
    "--output-dir", str(OUTPUT_DIR),
]
if ABLATIONS:
    command.extend(["--ablations", *ABLATIONS])
(OUTPUT_DIR / "command.json").write_text(json.dumps(command, indent=2))

log_path = OUTPUT_DIR / "training.log"
with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command, cwd=REPO, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
    finally:
        if process.poll() is None:
            process.terminate()
            process.wait(timeout=30)
if returncode:
    raise RuntimeError(f"Training failed with exit code {returncode}; see {log_path}")
print("Completed. Results saved to:", OUTPUT_DIR)

## Cell 7 — Read validation results

In [ ]:
import pandas as pd
from IPython.display import display

table = pd.read_csv(OUTPUT_DIR / "comparison.csv")
display(table[[
    "seed", "model", "filtered_mrr", "raw_mrr",
    "filtered_hits@1", "filtered_hits@10", "filtered_mean_rank",
    "best_epoch", "total_budget_updates", "elapsed_seconds",
]].sort_values(["seed", "filtered_mrr"], ascending=[True, False]))

report = json.loads((OUTPUT_DIR / "comparison.json").read_text())
assert report["manifest"]["test_evaluated"] is False
print("Aggregate validation results:")
print(json.dumps(report["aggregate"], indent=2))
print("Per-model reports, checkpoints, diagnostics and logs:", OUTPUT_DIR)
print("best_epoch=0 means the transferred baseline checkpoint remained best.")

## After the first run
Inspect validation curves, gradient/message norms and identity scales in the per-model reports before selecting ablations. If the result warrants replication, change SEEDS to [17, 23, 42], rerun Cell 2 to create a new output folder, and rerun the relevant cells.

Completed phases are checkpointed in Drive. The currently running phase is not resumable mid-epoch; an interrupted active phase must restart. The streamed log is flushed to Drive during execution.

If CUDA memory is exhausted, lower BATCH_SIZE to 1024, create a new output folder by rerunning Cell 2, and rerun the training cell. Keep the other comparison settings unchanged.

Keep test evaluation sealed during development. These cells were syntax-checked locally and their runner was smoke-tested on CPU; this notebook has not been executed on a hosted Colab GPU.